In [1]:
"""
(1) Turn the raw climate tables (02) into anomaly features and SPI.
(2) Lag analysis: does climate at t-h predict the NDVI stress label at t?
"""
import argparse
import glob
import os
import sys

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import gamma, norm
from sklearn.metrics import roc_auc_score

# ------------------------- CONFIG -------------------------
VARS = ["chirps", "dry", "sm1", "sm2", "t2m", "vpd", "lst"]
NK = 23                                   # windows per year
BASE = (2001, 2020)
MIN_BASE_N = 10
SPI_SCALES = {2: "spi32", 4: "spi64", 6: "spi96"}    # windows -> name (days)
# sign so that "larger = more stress-like" for AUC
STRESS_SIGN = {"chirps": -1, "dry": 1, "sm1": -1, "sm2": -1, "t2m": 1, "vpd": 1, "lst": 1,
               "spi32": -1, "spi64": -1, "spi96": -1}
STRESS_Z = -1.0
CELL_DEG = 0.05


# ------------------------- loading -------------------------
def load_climate(clim_dir):
    files = sorted(glob.glob(os.path.join(clim_dir, "climate_*.csv")))
    if not files:
        raise FileNotFoundError(f"No climate_*.csv in {clim_dir}")
    df = pd.concat((pd.read_csv(f, dtype={"pid": str}) for f in files), ignore_index=True)
    long = pd.wide_to_long(
        df, stubnames=[f"{v}_k" for v in VARS], i=["pid", "year", "lon", "lat"], j="k"
    ).reset_index()
    long = long.rename(columns={f"{v}_k": v for v in VARS})
    long["k"] = long["k"].astype(int)
    long["t"] = long["year"] * NK + long["k"]
    ix = np.floor(long["lon"] / CELL_DEG).astype(int)
    iy = np.floor(long["lat"] / CELL_DEG).astype(int)
    long["cell"] = ix * 10000 + iy
    print(f"Climate: {len(files)} files, {long['pid'].nunique()} points, "
          f"{long['cell'].nunique()} CHIRPS cells, years {long['year'].min()}-{long['year'].max()}")
    return long


def load_label(label_dir):
    pq = os.path.join(label_dir, "label_long.parquet")
    csv = os.path.join(label_dir, "label_long.csv")
    lab = pd.read_parquet(pq) if os.path.exists(pq) else pd.read_csv(csv, dtype={"pid": str})
    lab["pid"] = lab["pid"].astype(str)
    lab["t"] = lab["year"] * NK + lab["k"]
    print(f"Label: {lab['pid'].nunique()} points, years {lab['year'].min()}-{lab['year'].max()}")
    return lab


# ------------------------- anomalies -------------------------
def zscore_loyo(df, col, keys, base=BASE, loyo=True, min_n=MIN_BASE_N):
    y0, y1 = base
    x = df[col].astype(float)
    inb = df["year"].between(y0, y1) & x.notna()
    tmp = df[keys].copy()
    tmp["x"] = x.where(inb)
    tmp["x2"] = tmp["x"] ** 2
    g = tmp.groupby(keys)["x"].agg(["count", "sum"]).join(tmp.groupby(keys)["x2"].sum().rename("ss"))
    m = df[keys].join(g, on=keys)
    n = m["count"].astype(float)
    s = m["sum"].astype(float)
    ss = m["ss"].astype(float)
    if loyo:
        n = n - inb.astype(float)
        s = s - x.where(inb, 0.0)
        ss = ss - (x ** 2).where(inb, 0.0)
    mu = s / n
    sd = np.sqrt(np.clip((ss - n * mu ** 2) / (n - 1), 0, None))
    floor = 0.05 * np.nanstd(x)
    z = (x - mu) / np.maximum(sd, floor)
    return z.where(n >= min_n)


# ------------------------- SPI (per cell) -------------------------
def lagged_values(series, key_arrays, t, h):
    """series indexed by (*keys, t); return its value at t-h for each row (NaN if missing)."""
    idx = pd.MultiIndex.from_arrays(list(key_arrays) + [np.asarray(t) - h])
    return series.reindex(idx).to_numpy()


def spi_loyo(cell_df, acc_col, base=BASE, loyo=True, min_pos=MIN_BASE_N):
    y0, y1 = base
    x = cell_df[acc_col].astype(float)
    inb = cell_df["year"].between(y0, y1) & x.notna()
    pos = inb & (x > 0)
    zero = inb & (x <= 0)
    lnx = np.log(x.where(pos, 1.0))
    tmp = cell_df[["cell", "k"]].copy()
    tmp["n"] = inb.astype(float)
    tmp["n0"] = zero.astype(float)
    tmp["npos"] = pos.astype(float)
    tmp["sx"] = x.where(pos, 0.0)
    tmp["sl"] = lnx
    g = tmp.groupby(["cell", "k"])[["n", "n0", "npos", "sx", "sl"]].sum()
    m = cell_df[["cell", "k"]].join(g, on=["cell", "k"])
    n, n0, npos, sx, sl = (m[c].astype(float) for c in ["n", "n0", "npos", "sx", "sl"])
    if loyo:
        n = n - inb.astype(float)
        n0 = n0 - zero.astype(float)
        npos = npos - pos.astype(float)
        sx = sx - x.where(pos, 0.0)
        sl = sl - lnx
    with np.errstate(divide="ignore", invalid="ignore"):
        p0 = n0 / n
        mean_pos = sx / npos
        A = np.clip(np.log(mean_pos) - sl / npos, 1e-4, None)
        a = (1 + np.sqrt(1 + 4 * A / 3)) / (4 * A)
        scale = mean_pos / a
        prob = p0 + (1 - p0) * gamma.cdf(np.clip(x, 0, None), a, scale=scale)
        spi = norm.ppf(np.clip(prob, 1e-6, 1 - 1e-6))
    spi = pd.Series(spi, index=cell_df.index)
    return spi.where((npos >= min_pos) & (n >= MIN_BASE_N) & x.notna())


def add_spi(long, base, loyo):
    cell = long.groupby(["cell", "year", "k"], as_index=False)["chirps"].mean()
    cell["t"] = cell["year"] * NK + cell["k"]
    cell = cell.sort_values(["cell", "t"]).reset_index(drop=True)
    s = cell.set_index(["cell", "t"])["chirps"]
    for n_win, name in SPI_SCALES.items():
        parts = [lagged_values(s, [cell["cell"]], cell["t"], j) for j in range(n_win)]
        cell[f"acc_{name}"] = np.sum(parts, axis=0)          # NaN if any window missing
        cell[name] = spi_loyo(cell.rename(columns={f"acc_{name}": "acc"}), "acc", base, loyo).to_numpy()
    keep = ["cell", "year", "k"] + list(SPI_SCALES.values())
    return long.merge(cell[keep], on=["cell", "year", "k"], how="left")


# ------------------------- features -------------------------
def build_features(clim, base=BASE, loyo=True):
    clim = clim.sort_values(["pid", "t"]).reset_index(drop=True)
    for v in VARS:
        clim[f"{v}_z"] = zscore_loyo(clim, v, ["pid", "k"], base, loyo)
        print(f"  anomaly done: {v}")
    clim = add_spi(clim, base, loyo)
    print("  SPI done")
    return clim


# ------------------------- lag analysis -------------------------
def lag_analysis(feat, label, max_lag=6):
    pred_cols = [f"{v}_z" for v in VARS] + list(SPI_SCALES.values())
    pid_cell = feat.drop_duplicates("pid").set_index("pid")["cell"]
    lab = label[label["pid"].isin(pid_cell.index)].copy()
    lab["cell"] = lab["pid"].map(pid_cell)

    # cell x zone x t aggregation
    zone_map = lab.drop_duplicates("pid").set_index("pid")["zone"]
    feat = feat[feat["pid"].isin(zone_map.index)].copy()
    feat["zone"] = feat["pid"].map(zone_map)
    pred = feat.groupby(["cell", "zone", "t"])[pred_cols].mean()
    tgt = lab.groupby(["cell", "zone", "t"]).agg(z=("z", "mean"), n_pts=("z", "count"))
    tgt = tgt[tgt["z"].notna()]
    print(f"Lag analysis on {len(tgt)} cell-window rows, {tgt.index.get_level_values('cell').nunique()} cells")

    cells = tgt.index.get_level_values("cell")
    zones = tgt.index.get_level_values("zone")
    t = tgt.index.get_level_values("t")
    y_bin = (tgt["z"] < STRESS_Z).to_numpy()

    rows = []
    for h in range(0, max_lag + 1):
        idx = pd.MultiIndex.from_arrays([cells, zones, np.asarray(t) - h])
        sh = pred.reindex(idx)
        sh.index = tgt.index
        sh["z"] = tgt["z"]
        r_all = sh[pred_cols].corrwith(sh["z"])
        for p in pred_cols:
            base_name = p[:-2] if p.endswith("_z") else p
            sign = STRESS_SIGN[base_name]
            xv = sh[p].to_numpy() * sign
            ok = ~np.isnan(xv)
            auc = np.nan
            if ok.sum() > 50 and len(np.unique(y_bin[ok])) == 2:
                auc = roc_auc_score(y_bin[ok], xv[ok])
            rows.append(dict(lag_windows=h, lag_days=16 * h, predictor=p, scope="all",
                             pearson_r=r_all[p], auc=auc, n=int(ok.sum())))
        for zn, g in sh.groupby(level="zone"):
            r_z = g[pred_cols].corrwith(g["z"])
            for p in pred_cols:
                rows.append(dict(lag_windows=h, lag_days=16 * h, predictor=p, scope=f"zone{int(zn)}",
                                 pearson_r=r_z[p], auc=np.nan, n=int(g[p].notna().sum())))
    return pd.DataFrame(rows)


def plot_lags(res, out_png):
    a = res[res["scope"] == "all"]
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
    for p, g in a.groupby("predictor"):
        ax[0].plot(g["lag_days"], g["pearson_r"], marker="o", ms=3, label=p)
        ax[1].plot(g["lag_days"], g["auc"], marker="o", ms=3, label=p)
    ax[0].axhline(0, color="k", lw=0.6)
    ax[0].set_title("Correlation with NDVI z (cell level)")
    ax[0].set_xlabel("Lead time (days, predictor leads NDVI)")
    ax[0].set_ylabel("Pearson r")
    ax[1].axhline(0.5, color="k", lw=0.6)
    ax[1].set_title("AUC for NDVI stress (cell-mean z < -1)")
    ax[1].set_xlabel("Lead time (days)")
    ax[1].set_ylabel("AUC (0.5 = no skill)")
    ax[1].legend(fontsize=7, ncol=2)
    fig.tight_layout()
    fig.savefig(out_png, dpi=150)
    plt.close(fig)


def summarise(res):
    a = res[res["scope"] == "all"].dropna(subset=["pearson_r"])
    print("\nStrongest |r| by predictor (lag in days):")
    best = a.loc[a.groupby("predictor")["pearson_r"].apply(lambda s: s.abs().idxmax())]
    print(best[["predictor", "lag_days", "pearson_r", "auc"]].round(3).to_string(index=False))
    print("\nBest AUC by predictor:")
    b = res[(res["scope"] == "all")].dropna(subset=["auc"])
    bb = b.loc[b.groupby("predictor")["auc"].idxmax()]
    print(bb[["predictor", "lag_days", "auc"]].round(3).to_string(index=False))


# ------------------------- main -------------------------
def save_table(df, path_no_ext):
    try:
        df.to_parquet(path_no_ext + ".parquet", index=False)
        print(f"Saved {path_no_ext}.parquet")
    except Exception as e:
        df.to_csv(path_no_ext + ".csv", index=False)
        print(f"parquet failed ({e}); saved {path_no_ext}.csv")


def run(clim_dir, label_dir, out_dir, base=BASE, loyo=True, max_lag=6):
    os.makedirs(out_dir, exist_ok=True)
    label = load_label(label_dir)
    clim = load_climate(clim_dir)
    clim = clim[clim["pid"].isin(label["pid"].unique())]
    feat = build_features(clim, base, loyo)
    save_table(feat.drop(columns=["t"]), os.path.join(out_dir, "features_long"))

    res = lag_analysis(feat, label, max_lag)
    res.to_csv(os.path.join(out_dir, "lag_results.csv"), index=False)
    plot_lags(res, os.path.join(out_dir, "lag_curves.png"))
    summarise(res)
    return feat, res


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--clim-dir", required=True)
    ap.add_argument("--label-dir", required=True)
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--base-start", type=int, default=BASE[0])
    ap.add_argument("--base-end", type=int, default=BASE[1])
    ap.add_argument("--no-loyo", action="store_true")
    ap.add_argument("--max-lag", type=int, default=6, help="max lag in 16-day windows")
    a = ap.parse_args(argv)
    return run(a.clim_dir, a.label_dir, a.out_dir, (a.base_start, a.base_end), not a.no_loyo, a.max_lag)


if __name__ == "__main__" and "ipykernel" not in sys.modules:
    main()

In [3]:
main(["--clim-dir", r"crop_stress\climate",
      "--label-dir", r"crop_stress\label",
      "--out-dir",   r"crop_stress\features"])

Label: 3185 points, years 2001.0-2025.0
Climate: 25 files, 8387 points, 869 CHIRPS cells, years 2001-2025
  anomaly done: chirps
  anomaly done: dry
  anomaly done: sm1
  anomaly done: sm2
  anomaly done: t2m
  anomaly done: vpd
  anomaly done: lst
  SPI done
Saved crop_stress\features\features_long.parquet
Lag analysis on 115297 cell-window rows, 572 cells

Strongest |r| by predictor (lag in days):
predictor  lag_days  pearson_r   auc
 chirps_z        16      0.168 0.563
    dry_z        16     -0.134 0.556
    lst_z         0     -0.358 0.671
    sm1_z         0      0.222 0.607
    sm2_z         0      0.237 0.611
    spi32         0      0.187 0.582
    spi64         0      0.211 0.590
    spi96         0      0.190 0.579
    t2m_z         0     -0.220 0.607
    vpd_z        16     -0.244 0.620

Best AUC by predictor:
predictor  lag_days   auc
 chirps_z        16 0.563
    dry_z        16 0.556
    lst_z         0 0.671
    sm1_z         0 0.607
    sm2_z         0 0.611
    spi32 

(          pid  year       lon       lat   k     chirps  dry       sm1  \
 0         105  2001  2.737942  6.969793   0   3.497316   15  0.107035   
 1         105  2001  2.737942  6.969793   1   0.000000   16  0.092514   
 2         105  2001  2.737942  6.969793   2   5.478588   15  0.071417   
 3         105  2001  2.737942  6.969793   3  16.375793   12  0.103606   
 4         105  2001  2.737942  6.969793   4  24.936403    8  0.196195   
 ...       ...   ...       ...       ...  ..        ...  ...       ...   
 1831370  8385  2025  3.306606  8.311457  18  61.393717    9  0.354122   
 1831371  8385  2025  3.306606  8.311457  19  16.120694   13  0.290613   
 1831372  8385  2025  3.306606  8.311457  20   3.000568   16  0.234038   
 1831373  8385  2025  3.306606  8.311457  21  10.832093   12  0.208647   
 1831374  8385  2025  3.306606  8.311457  22   0.000000   13  0.192147   
 
               sm2        t2m  ...  chirps_z     dry_z     sm1_z     sm2_z  \
 0        0.094344  27.424453  .